In [2]:
# ==========================================
# BEFORE TRAINING – BASE MODEL EVALUATION
# ==========================================
import os
import time
import json
import torch
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ==========================================
# DEVICE SETUP (CUDA → CPU FALLBACK)
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    print(f"🚀 Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("🛡️ Using CPU")

# ==========================================
# CONFIG
# ==========================================
base_model_id = "facebook/nllb-200-distilled-600M"
test_file = "translation_test_pairs_50.json"

SRC_KEY = "tamil"
TGT_KEY = "english"
SRC_LANG = "tam_Taml"
TGT_LANG = "eng_Latn"

# ==========================================
# LOAD MODEL
# ==========================================
print("⏳ Loading model...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(base_model_id).to(device)
model.eval()

bleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

# ==========================================
# LOAD & NORMALIZE DATA
# ==========================================
with open(test_file, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# 🔧 HANDLE MULTIPLE JSON FORMATS
if isinstance(raw_data, dict) and "pairs" in raw_data:
    data = raw_data["pairs"]
elif isinstance(raw_data, list):
    data = raw_data
else:
    raise ValueError("❌ Unsupported JSON structure")

print(f"✅ Loaded {len(data)} test samples")

# ==========================================
# EVALUATION LOOP
# ==========================================
predictions = []
references = []
total_time = 0.0
valid_count = 0

for i, row in enumerate(data):
    # 🛡️ SAFETY CHECK
    if not isinstance(row, dict):
        print(f"⚠️ Skipping row {i}: not a dict")
        continue

    if SRC_KEY not in row or TGT_KEY not in row:
        print(f"⚠️ Skipping row {i}: missing keys {row.keys()}")
        continue

    src_text = row[SRC_KEY]
    tgt_text = row[TGT_KEY]

    tokenizer.src_lang = SRC_LANG
    inputs = tokenizer(
        src_text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    # ⏱ Accurate timing (GPU-safe)
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.time()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(TGT_LANG),
            max_length=128,
            num_beams=4
        )

    if device == "cuda":
        torch.cuda.synchronize()
    end = time.time()

    total_time += (end - start)

    pred = tokenizer.decode(output[0], skip_special_tokens=True)
    predictions.append(pred)
    references.append([tgt_text])
    valid_count += 1

    if i % 10 == 0:
        print(f"   Processed {i}/{len(data)}")

# ==========================================
# METRICS
# ==========================================
if valid_count == 0:
    raise RuntimeError("❌ No valid samples to evaluate")

bleu_score = bleu.compute(
    predictions=predictions,
    references=references
)["score"]

chrf_score = chrf.compute(
    predictions=predictions,
    references=references
)["score"]

avg_latency = total_time / valid_count

print("\n===== BEFORE TRAINING RESULTS =====")
print(f"Samples Evaluated : {valid_count}")
print(f"BLEU              : {bleu_score:.2f}")
print(f"CHRF++            : {chrf_score:.2f}")
print(f"Avg Latency (sec) : {avg_latency:.3f}")
print("==================================")


🚀 Using GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
⏳ Loading model...
✅ Loaded 60 test samples
   Processed 0/60
   Processed 10/60
   Processed 20/60
   Processed 30/60
   Processed 40/60
   Processed 50/60

===== BEFORE TRAINING RESULTS =====
Samples Evaluated : 60
BLEU              : 30.23
CHRF++            : 52.90
Avg Latency (sec) : 2.707


In [3]:
# ==========================================
# ENGLISH → TAMIL EVALUATION
# ==========================================
SRC_KEY_REV = "english"
TGT_KEY_REV = "tamil"
SRC_LANG_REV = "eng_Latn"
TGT_LANG_REV = "tam_Taml"

print("\n🔁 Evaluating English → Tamil")

predictions_rev = []
references_rev = []
total_time_rev = 0.0
valid_count_rev = 0

for i, row in enumerate(data):
    # Safety checks
    if not isinstance(row, dict):
        continue
    if SRC_KEY_REV not in row or TGT_KEY_REV not in row:
        continue

    src_text = row[SRC_KEY_REV]
    tgt_text = row[TGT_KEY_REV]

    tokenizer.src_lang = SRC_LANG_REV
    inputs = tokenizer(
        src_text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    # Accurate timing
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.time()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(TGT_LANG_REV),
            max_length=128,
            num_beams=4
        )

    if device == "cuda":
        torch.cuda.synchronize()
    end = time.time()

    total_time_rev += (end - start)

    pred = tokenizer.decode(output[0], skip_special_tokens=True)
    predictions_rev.append(pred)
    references_rev.append([tgt_text])
    valid_count_rev += 1
bleu_score_rev = bleu.compute(
    predictions=predictions_rev,
    references=references_rev
)["score"]

chrf_score_rev = chrf.compute(
    predictions=predictions_rev,
    references=references_rev
)["score"]

avg_latency_rev = total_time_rev / valid_count_rev

print("\n===== ENGLISH → TAMIL RESULTS =====")
print(f"Samples Evaluated : {valid_count_rev}")
print(f"BLEU              : {bleu_score_rev:.2f}")
print(f"CHRF++            : {chrf_score_rev:.2f}")
print(f"Avg Latency (sec) : {avg_latency_rev:.3f}")
print("==================================")



🔁 Evaluating English → Tamil

===== ENGLISH → TAMIL RESULTS =====
Samples Evaluated : 60
BLEU              : 23.65
CHRF++            : 61.13
Avg Latency (sec) : 3.271


In [10]:
evaluate_bidirectional(
    "facebook/nllb-200-1.3B", "nllb",
    "tam_Taml", "eng_Latn",
    src_ta2en, ref_ta2en,
    csv_prefix="nllb1300_ta2en"
)






NameError: name 'src_ta2en' is not defined

In [18]:
evaluate_bidirectional(
    "facebook/m2m100_418M", "m2m100",
    "en", "ta",
    src_en2ta, ref_en2ta,
    csv_prefix="m2m100_en2ta"
)



===== Evaluating facebook/m2m100_418M =====
Saved m2m100_en2ta_predictions.csv & m2m100_en2ta_metrics.csv


{'BLEU': 2.7602807896927186,
 'chrF': 29.22789291150621,
 'Exact': 0.0,
 'Time/sent': 5.2597041238438}

In [19]:
evaluate_bidirectional(
    "facebook/nllb-200-distilled-600M", "nllb",
    "eng_Latn", "tam_Taml",
    src_en2ta, ref_en2ta,
    csv_prefix="nllb600_en2ta"
)



===== Evaluating facebook/nllb-200-distilled-600M =====
Saved nllb600_en2ta_predictions.csv & nllb600_en2ta_metrics.csv


{'BLEU': 0.2598757121984294,
 'chrF': 1.5829642589090902,
 'Exact': 0.0,
 'Time/sent': 7.243316555565054}

In [ ]:
evaluate_bidirectional(
    "facebook/nllb-200-1.3B", "nllb",
    "eng_Latn", "tam_Taml",
    src_en2ta, ref_en2ta,
    csv_prefix="nllb1300_en2ta"
)



===== Evaluating facebook/nllb-200-1.3B =====
Saved nllb1300_en2ta_predictions.csv & nllb1300_en2ta_metrics.csv


{'BLEU': 0.09787912652915184,
 'chrF': 0.2735291463634723,
 'Exact': 0.0,
 'Time/sent': 11.164155299013311}

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

def translate_en_to_ta(text):
    tokenizer.src_lang = SRC
    enc = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    with torch.no_grad():
        if FAMILY == "m2m100":
            bos = tokenizer.get_lang_id(TGT)
            out = model.generate(**enc, forced_bos_token_id=bos, max_length=256)
        else:
            # NLLB: no BOS forcing
            out = model.generate(**enc, max_length=256)

    return tokenizer.decode(out[0], skip_special_tokens=True).strip()


Using: cpu


In [ ]:
for item in sample_data:
    print(f"\nID {item['id']}")
    print("English:", item["english"])
    
    pred = translate_en_to_ta(item["english"])
    print("Model Tamil:", pred)
    
    print("Reference Tamil:", item["tamil"])



ID 61
English: If a person submits false income information to obtain government assistance, it is considered an act of cheating. This behaviour is treated as misuse of government resources and is punishable under the law.
Model Tamil: If a person submits false income information to obtain government assistance, it is considered an act of cheating.இந்த நடவடிக்கை அரசாங்க வளங்கள் தவறான பயன்பாடு மற்றும் சட்டத்தின் கீழ் வேதனையாகும்.
Reference Tamil: ஒரு நபர் அரசு உதவித் தொகை பெறுவதற்காக வருமானத்தைப் பற்றிய பொய்யான தகவலை சமர்ப்பித்தால், அது மோசடி குற்றமாக கருதப்படும். இந்த செயல் அரசு வளங்களை தவறாக பயன்படுத்துவதாகவும் சட்டம் அதை தண்டிக்கத்தக்க குற்றமாகவும் பார்க்கிறது.

ID 62
English: If a person knowingly keeps stolen property at home, they are considered guilty if it can be proven that they knew it was stolen. This applies even to those who were not involved in the theft itself.
Model Tamil: ஒரு மனிதன் வீட்டில் சுட்டுக் கொல்லப்பட்ட பொருட்களை அறிந்து வைத்தால், அவர்களை குற்றவாளிகளாகக் கருதப

In [27]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
FAMILY = "nllb"

SRC = "eng_Latn"
TGT = "tam_Taml"


In [34]:
# !pip install transformers sentencepiece sacrebleu torch tqdm pandas nltk

import json
import pandas as pd
import torch
import time
import sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cpu


In [36]:
def evaluate_bidirectional(
    model_name, family,
    src_lang_code, tgt_lang_code,
    src_texts, ref_texts,
    csv_prefix="output"
):
    print(f"\n===== Evaluating {model_name} =====")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    model.eval()

    predictions = []
    start = time.time()

    for text in tqdm(src_texts):
        if family == "m2m100":
            tokenizer.src_lang = src_lang_code
        enc = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

        with torch.no_grad():
            if family == "m2m100":
                bos = tokenizer.get_lang_id(tgt_lang_code)
                out = model.generate(
                    **enc,
                    forced_bos_token_id=bos,
                    max_length=256,
                    num_beams=4
                )
            else:
                # NLLB
                out = model.generate(
                    **enc,
                    max_length=256,
                    num_beams=4
                )

        decoded = tokenizer.decode(out[0], skip_special_tokens=True).strip()
        predictions.append(decoded)

    # Metrics
    elapsed = time.time() - start
    sec_per_sentence = elapsed / len(src_texts)
    bleu = sacrebleu.corpus_bleu(predictions, [ref_texts]).score
    chrf = sacrebleu.corpus_chrf(predictions, [ref_texts]).score
    exact = sum(1 for p, r in zip(predictions, ref_texts) if p == r) / len(ref_texts) * 100

    # Save CSV
    pd.DataFrame({
        "Source": src_texts,
        "Reference": ref_texts,
        "Predicted": predictions,
    }).to_csv(f"{csv_prefix}_predictions.csv", index=False)

    pd.DataFrame([{
        "Model": model_name,
        "BLEU": bleu,
        "chrF": chrf,
        "ExactMatch(%)": exact,
        "SecondsPerSentence": sec_per_sentence
    }]).to_csv(f"{csv_prefix}_metrics.csv", index=False)

    print(f"Saved {csv_prefix}_predictions.csv & {csv_prefix}_metrics.csv")
    return {"BLEU": bleu, "chrF": chrf, "Exact": exact, "Time/sent": sec_per_sentence}


In [38]:
result_m2m100_ta2en = evaluate_bidirectional(
    "facebook/m2m100_418M", "m2m100",
    src_lang_code="ta", tgt_lang_code="en",
    src_texts=src_ta2en, ref_texts=ref_ta2en,
    csv_prefix="m2m100_ta2en"
)

print(result_m2m100_ta2en)



===== Evaluating facebook/m2m100_418M =====


100%|██████████| 88/88 [07:08<00:00,  4.86s/it]


Saved m2m100_ta2en_predictions.csv & m2m100_ta2en_metrics.csv
{'BLEU': 5.8170255473213155, 'chrF': 27.56397560469236, 'Exact': 0.0, 'Time/sent': 4.864628125320781}


In [45]:
# !pip install transformers sentencepiece torch pandas tqdm bert-score

import json
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import time
from bert_score import score  # semantic evaluation
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cpu


In [40]:
 !pip install transformers sentencepiece torch pandas tqdm bert-score


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
json_path = "D://reserach-testing//translation_test_pairs_50.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

pairs = data.get("pairs", [])
if not pairs:
    raise ValueError("No 'pairs' found in JSON.")

# Normalize keys
for i in range(len(pairs)):
    pairs[i] = {k.strip().lower(): v for k, v in pairs[i].items()}

# Detect tamil and english keys dynamically
tamil_key = next((k for k in pairs[0].keys() if "tamil" in k), None)
english_key = next((k for k in pairs[0].keys() if "english" in k), None)
if not tamil_key or not english_key:
    raise KeyError(f"Cannot find Tamil or English keys. Available: {pairs[0].keys()}")

# Prepare bidirectional lists
src_ta2en, ref_ta2en = [], []
src_en2ta, ref_en2ta = [], []

for p in pairs:
    ta = p.get(tamil_key, "").strip()
    en = p.get(english_key, "").strip()
    if ta and en:
        src_ta2en.append(ta)
        ref_ta2en.append(en)
        src_en2ta.append(en)
        ref_en2ta.append(ta)

print("TA→EN samples:", len(src_ta2en))
print("EN→TA samples:", len(src_en2ta))


TA→EN samples: 88
EN→TA samples: 88


In [47]:
def evaluate_model_bert(
    model_name, family,
    src_lang_code, tgt_lang_code,
    src_texts, ref_texts,
    csv_prefix="output"
):
    print(f"\n===== Evaluating {model_name} =====")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    model.eval()

    predictions = []
    start = time.time()

    for text in tqdm(src_texts):
        if family == "m2m100":
            tokenizer.src_lang = src_lang_code
        enc = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

        with torch.no_grad():
            if family == "m2m100":
                bos = tokenizer.get_lang_id(tgt_lang_code)
                out = model.generate(
                    **enc,
                    forced_bos_token_id=bos,
                    max_length=256,
                    num_beams=4
                )
            else:
                out = model.generate(
                    **enc,
                    max_length=256,
                    num_beams=4
                )

        decoded = tokenizer.decode(out[0], skip_special_tokens=True).strip()
        predictions.append(decoded)

    # --- Semantic evaluation with BERTScore ---
    P, R, F1 = score(predictions, ref_texts, lang="en" if "en" in tgt_lang_code else "xx", verbose=True)
    avg_f1 = F1.mean().item()

    elapsed = time.time() - start
    sec_per_sentence = elapsed / len(src_texts)

    print(f"Average BERTScore F1: {avg_f1:.4f}, Time per sentence: {sec_per_sentence:.2f}s")

    # Save CSV
    pd.DataFrame({
        "Source": src_texts,
        "Reference": ref_texts,
        "Predicted": predictions,
    }).to_csv(f"{csv_prefix}_predictions.csv", index=False)

    pd.DataFrame([{
        "Model": model_name,
        "BERTScore_F1": avg_f1,
        "SecondsPerSentence": sec_per_sentence
    }]).to_csv(f"{csv_prefix}_metrics.csv", index=False)

    print(f"Saved {csv_prefix}_predictions.csv & {csv_prefix}_metrics.csv")
    return {"BERTScore_F1": avg_f1, "Time/sent": sec_per_sentence}


In [48]:
result_nllb600_en2ta = evaluate_model_bert(
    "facebook/nllb-200-distilled-600M", "nllb",
    src_lang_code="eng_Latn", tgt_lang_code="tam_Taml",
    src_texts=src_en2ta, ref_texts=ref_en2ta,
    csv_prefix="nllb600_en2ta_bertscore"
)

print(result_nllb600_en2ta)



===== Evaluating facebook/nllb-200-distilled-600M =====


100%|██████████| 88/88 [13:55<00:00,  9.49s/it]
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:  37%|###6      | 419M/1.13G [00:00<?, ?B/s]

calculating scores...
computing bert embedding.


  0%|          | 0/3 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 4.37 seconds, 20.14 sentences/sec
Average BERTScore F1: 0.6487, Time per sentence: 9.93s
Saved nllb600_en2ta_bertscore_predictions.csv & nllb600_en2ta_bertscore_metrics.csv
{'BERTScore_F1': 0.6487048268318176, 'Time/sent': 9.932326538996262}


In [49]:
result_m2m100_en2ta = evaluate_model_bert(
    "facebook/m2m100_418M", "m2m100",
    src_lang_code="en", tgt_lang_code="ta",
    src_texts=src_en2ta, ref_texts=ref_en2ta,
    csv_prefix="m2m100_en2ta_bertscore"
)

print(result_m2m100_en2ta)



===== Evaluating facebook/m2m100_418M =====


100%|██████████| 88/88 [17:54<00:00, 12.22s/it] 


calculating scores...
computing bert embedding.


  0%|          | 0/3 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 8.43 seconds, 10.44 sentences/sec
Average BERTScore F1: 0.7412, Time per sentence: 12.33s
Saved m2m100_en2ta_bertscore_predictions.csv & m2m100_en2ta_bertscore_metrics.csv
{'BERTScore_F1': 0.7412149906158447, 'Time/sent': 12.326902622526342}


In [50]:
result_nllb1300_en2ta = evaluate_model_bert(
    "facebook/nllb-200-1.3B", "nllb",
    src_lang_code="eng_Latn", tgt_lang_code="tam_Taml",
    src_texts=src_en2ta, ref_texts=ref_en2ta,
    csv_prefix="nllb1300_en2ta_bertscore"
)

print(result_nllb1300_en2ta)



===== Evaluating facebook/nllb-200-1.3B =====


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:  47%|####7     | 4.95G/10.4G [00:00<?, ?B/s]

100%|██████████| 88/88 [18:34<00:00, 12.67s/it]


calculating scores...
computing bert embedding.


  0%|          | 0/3 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 5.70 seconds, 15.43 sentences/sec
Average BERTScore F1: 0.6562, Time per sentence: 12.77s
Saved nllb1300_en2ta_bertscore_predictions.csv & nllb1300_en2ta_bertscore_metrics.csv
{'BERTScore_F1': 0.6562349200248718, 'Time/sent': 12.765515844930302}


In [51]:
from bert_score import score

cands = ["நான் பள்ளி செல்கிறேன்"]
refs = ["நான் பள்ளிக்கு போகிறேன்"]

P, R, F1 = score(cands, refs, lang='ta', verbose=True)
print(F1.mean().item())


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.10 seconds, 9.74 sentences/sec
0.8927012085914612


In [52]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sacrebleu import corpus_bleu

# Load NLLB-600M model (English → Tamil)
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="eng_Latn", tgt_lang="tam_Taml")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Sample sentences
english_sentences = [
    "The accused stole money.",
    "He committed murder.",
    "The court sentenced him to five years.",
    "She was charged with fraud.",
    "The police arrested the suspect.",
    "He violated the law.",
    "The witness testified in court.",
    "The theft occurred at night.",
    "He is innocent until proven guilty.",
    "The contract was signed by both parties."
]

# Reference Tamil translations
references = [
    ["குற்றவாளி பணத்தை திருடினார்."],
    ["அவர் கொலை செய்தார்."],
    ["நீதிமன்றம் அவருக்கு ஐந்து வருடங்கள் தீர்ப்பிட்டது."],
    ["அவர் மோசடி குற்றச்சாட்டில் சிக்கினார்."],
    ["போலீசார் சந்தேகப்படுத்தப்பட்டவரை கைது செய்தனர்."],
    ["அவர் சட்டத்தை மீறினார்."],
    ["சாட்சி நீதிமன்றத்தில் சாட்சியம் அளித்தார்."],
    ["திருட்டு இரவில் நடந்தது."],
    ["அவர் குற்றம்சாட்டபட்டவராக நிரூபிக்கப்படும்வரை निर्दोषன்."],
    ["ஒப்பந்தம் இரு தரப்பினராலும் கையெழுத்து செய்யப்பட்டது."]
]

# Translate English → Tamil
translations = []
for sent in english_sentences:
    inputs = tokenizer(sent, return_tensors="pt")
    output_ids = model.generate(**inputs)
    translated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    translations.append(translated)

# Calculate BLEU
bleu = corpus_bleu(translations, references)
print("BLEU score:", bleu.score)

# Show translations
for en, ref, pred in zip(english_sentences, references, translations):
    print(f"EN: {en}\nRef: {ref[0]}\nPred: {pred}\n")


BLEU score: 15.97357760615681
EN: The accused stole money.
Ref: குற்றவாளி பணத்தை திருடினார்.
Pred: Suçlu para çaldı.

EN: He committed murder.
Ref: அவர் கொலை செய்தார்.
Pred: Έκανε φόνο.

EN: The court sentenced him to five years.
Ref: நீதிமன்றம் அவருக்கு ஐந்து வருடங்கள் தீர்ப்பிட்டது.
Pred: Mahkeme onu beş yıl hapse mahkûm etti.

EN: She was charged with fraud.
Ref: அவர் மோசடி குற்றச்சாட்டில் சிக்கினார்.
Pred: E' stata accusata di frode.

EN: The police arrested the suspect.
Ref: போலீசார் சந்தேகப்படுத்தப்பட்டவரை கைது செய்தனர்.
Pred: Polis şüpheliyi tutukladı.

EN: He violated the law.
Ref: அவர் சட்டத்தை மீறினார்.
Pred: Kanunu ihlal etti.

EN: The witness testified in court.
Ref: சாட்சி நீதிமன்றத்தில் சாட்சியம் அளித்தார்.
Pred: Il testimone ha testimoniato in tribunale.

EN: The theft occurred at night.
Ref: திருட்டு இரவில் நடந்தது.
Pred: Hırsızlık gece oldu.

EN: He is innocent until proven guilty.
Ref: அவர் குற்றம்சாட்டபட்டவராக நிரூபிக்கப்படும்வரை निर्दोषன்.
Pred: E nevinovat până cân

In [53]:
# Install dependencies if not already installed
# pip install transformers sacrebleu torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sacrebleu import corpus_bleu

# ===============================
# Load NLLB-600M model
# ===============================
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="eng_Latn", tgt_lang="tam_Taml")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ===============================
# Sample English sentences
# ===============================
english_sentences = [
    "The accused stole money.",
    "He committed murder.",
    "The court sentenced him to five years.",
    "She was charged with fraud.",
    "The police arrested the suspect.",
    "He violated the law.",
    "The witness testified in court.",
    "The theft occurred at night.",
    "He is innocent until proven guilty.",
    "The contract was signed by both parties."
]

# Reference Tamil translations
references = [
    ["குற்றவாளி பணத்தை திருடினார்."],
    ["அவர் கொலை செய்தார்."],
    ["நீதிமன்றம் அவருக்கு ஐந்து வருடங்கள் தீர்ப்பிட்டது."],
    ["அவர் மோசடி குற்றச்சாட்டில் சிக்கினார்."],
    ["போலீசார் சந்தேகப்படுத்தப்பட்டவரை கைது செய்தனர்."],
    ["அவர் சட்டத்தை மீறினார்."],
    ["சாட்சி நீதிமன்றத்தில் சாட்சியம் அளித்தார்."],
    ["திருட்டு இரவில் நடந்தது."],
    ["அவர் குற்றம்சாட்டபட்டவராக நிரூபிக்கப்படும்வரை निर्दोषன்."],
    ["ஒப்பந்தம் இரு தரப்பினராலும் கையெழுத்து செய்யப்பட்டது."]
]

# ===============================
# Translate English → Tamil
# ===============================
translations = []
for sent in english_sentences:
    inputs = tokenizer(sent, return_tensors="pt")
    
    # Force Tamil output
    forced_bos_token_id = tokenizer.lang_code_to_id["tam_Taml"]
    output_ids = model.generate(**inputs, forced_bos_token_id=forced_bos_token_id)
    
    translated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    translations.append(translated)

# ===============================
# Calculate BLEU score
# ===============================
bleu = corpus_bleu(translations, references)
print("BLEU score:", bleu.score)

# ===============================
# Show translations
# ===============================
for en, ref, pred in zip(english_sentences, references, translations):
    print(f"EN: {en}\nRef: {ref[0]}\nPred: {pred}\n")


AttributeError: NllbTokenizerFast has no attribute lang_code_to_id

In [54]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sacrebleu import corpus_bleu
import torch

# ===============================
# Load NLLB-600M model
# ===============================
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="eng_Latn", tgt_lang="tam_Taml")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ===============================
# Sample English sentences
# ===============================
english_sentences = [
    "The accused stole money.",
    "He committed murder.",
    "The court sentenced him to five years.",
    "She was charged with fraud.",
    "The police arrested the suspect.",
    "He violated the law.",
    "The witness testified in court.",
    "The theft occurred at night.",
    "He is innocent until proven guilty.",
    "The contract was signed by both parties."
]

# Reference Tamil translations
references = [
    ["குற்றவாளி பணத்தை திருடினார்."],
    ["அவர் கொலை செய்தார்."],
    ["நீதிமன்றம் அவருக்கு ஐந்து வருடங்கள் தீர்ப்பிட்டது."],
    ["அவர் மோசடி குற்றச்சாட்டில் சிக்கினார்."],
    ["போலீசார் சந்தேகப்படுத்தப்பட்டவரை கைது செய்தனர்."],
    ["அவர் சட்டத்தை மீறினார்."],
    ["சாட்சி நீதிமன்றத்தில் சாட்சியம் அளித்தார்."],
    ["திருட்டு இரவில் நடந்தது."],
    ["அவர் குற்றம்சாட்டபட்டவராக நிரூபிக்கப்படும்வரை निर्दोषன்."],
    ["ஒப்பந்தம் இரு தரப்பினராலும் கையெழுத்து செய்யப்பட்டது."]
]

# ===============================
# Translate English → Tamil
# ===============================
translations = []
for sent in english_sentences:
    inputs = tokenizer(sent, return_tensors="pt")
    
    # Force Tamil output correctly
    forced_bos_token_id = tokenizer.get_lang_id("tam_Taml")
    output_ids = model.generate(**inputs, forced_bos_token_id=forced_bos_token_id)
    
    translated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    translations.append(translated)

# ===============================
# Calculate BLEU score
# ===============================
bleu = corpus_bleu(translations, references)
print("BLEU score:", bleu.score)

# ===============================
# Show translations
# ===============================
for en, ref, pred in zip(english_sentences, references, translations):
    print(f"EN: {en}\nRef: {ref[0]}\nPred: {pred}\n")


AttributeError: NllbTokenizerFast has no attribute get_lang_id

In [55]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sacrebleu import corpus_bleu
import torch

# Load model and tokenizer
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="eng_Latn", tgt_lang="tam_Taml")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Sample sentences
english_sentences = [
    "The accused stole money.",
    "He committed murder.",
    "The court sentenced him to five years.",
    "She was charged with fraud.",
    "The police arrested the suspect.",
    "He violated the law.",
    "The witness testified in court.",
    "The theft occurred at night.",
    "He is innocent until proven guilty.",
    "The contract was signed by both parties."
]

# Reference Tamil translations
references = [
    ["குற்றவாளி பணத்தை திருடினார்."],
    ["அவர் கொலை செய்தார்."],
    ["நீதிமன்றம் அவருக்கு ஐந்து வருடங்கள் தீர்ப்பிட்டது."],
    ["அவர் மோசடி குற்றச்சாட்டில் சிக்கினார்."],
    ["போலீசார் சந்தேகப்படுத்தப்பட்டவரை கைது செய்தனர்."],
    ["அவர் சட்டத்தை மீறினார்."],
    ["சாட்சி நீதிமன்றத்தில் சாட்சியம் அளித்தார்."],
    ["திருட்டு இரவில் நடந்தது."],
    ["அவர் குற்றம்சாட்டபட்டவராக நிரூபிக்கப்படும்வரை निर्दोषன்."],
    ["ஒப்பந்தம் இரு தரப்பினராலும் கையெழுத்து செய்யப்பட்டது."]
]

# Map language codes to token IDs manually (from NLLB documentation)
# Tamil code: "tam_Taml" = 0x1181 ? Actually we use the tokenizer's id via tokenizer.lang_code_to_id (older)
# Since newer versions removed this, we can set the bos token manually using tokenizer.lang_code_to_id.get()
# Alternative: directly set `forced_bos_token_id=tokenizer.convert_tokens_to_ids("<tam_Taml>")`

# Check special tokens
print("Tokenizer special tokens:", tokenizer.special_tokens_map)

# Translate English → Tamil
translations = []
for sent in english_sentences:
    inputs = tokenizer(sent, return_tensors="pt")
    
    # Use language token manually
    tam_token_id = tokenizer.convert_tokens_to_ids("<tam_Taml>")  # correct way in latest transformers
    output_ids = model.generate(**inputs, forced_bos_token_id=tam_token_id)
    
    translated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    translations.append(translated)

# Compute BLEU
bleu = corpus_bleu(translations, references)
print("BLEU score:", bleu.score)

# Show translations
for en, ref, pred in zip(english_sentences, references, translations):
    print(f"EN: {en}\nRef: {ref[0]}\nPred: {pred}\n")


Tokenizer special tokens: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>', 'additional_special_tokens': ['ace_Arab', 'ace_Latn', 'acm_Arab', 'acq_Arab', 'aeb_Arab', 'afr_Latn', 'ajp_Arab', 'aka_Latn', 'amh_Ethi', 'apc_Arab', 'arb_Arab', 'ars_Arab', 'ary_Arab', 'arz_Arab', 'asm_Beng', 'ast_Latn', 'awa_Deva', 'ayr_Latn', 'azb_Arab', 'azj_Latn', 'bak_Cyrl', 'bam_Latn', 'ban_Latn', 'bel_Cyrl', 'bem_Latn', 'ben_Beng', 'bho_Deva', 'bjn_Arab', 'bjn_Latn', 'bod_Tibt', 'bos_Latn', 'bug_Latn', 'bul_Cyrl', 'cat_Latn', 'ceb_Latn', 'ces_Latn', 'cjk_Latn', 'ckb_Arab', 'crh_Latn', 'cym_Latn', 'dan_Latn', 'deu_Latn', 'dik_Latn', 'dyu_Latn', 'dzo_Tibt', 'ell_Grek', 'eng_Latn', 'epo_Latn', 'est_Latn', 'eus_Latn', 'ewe_Latn', 'fao_Latn', 'pes_Arab', 'fij_Latn', 'fin_Latn', 'fon_Latn', 'fra_Latn', 'fur_Latn', 'fuv_Latn', 'gla_Latn', 'gle_Latn', 'glg_Latn', 'grn_Latn', 'guj_Gujr', 'hat_Latn', 'hau_Latn', '

In [56]:
# Install dependencies if needed:
# pip install transformers torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# -------------------------------
# Load NLLB-600M model
# -------------------------------
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="eng_Latn", tgt_lang="tam_Taml")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# -------------------------------
# English sentence to translate
# -------------------------------
sentence = "The accused stole money."

# Tokenize input
inputs = tokenizer(sentence, return_tensors="pt").to(device)

# Get Tamil token ID for forced language output
tam_token_id = tokenizer.convert_tokens_to_ids("<tam_Taml>")

# Generate translation
output_ids = model.generate(
    **inputs,
    forced_bos_token_id=tam_token_id,
    max_length=50,
    num_beams=5,
    early_stopping=True
)

# Decode output
translated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("English:", sentence)
print("Tamil  :", translated)


English: The accused stole money.
Tamil  : The accused stole money


In [57]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load model and tokenizer
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# Sentence
sentence = "The accused stole money."

# Tokenize with target language
inputs = tokenizer(sentence, return_tensors="pt").to(device)

# Generate translation with `tgt_lang_id`
# Get Tamil language code ID from tokenizer config
tgt_lang_id = tokenizer.lang_code_to_id["tam_Taml"]

output_ids = model.generate(
    **inputs,
    max_length=50,
    num_beams=5,
    early_stopping=True,
    forced_bos_token_id=tgt_lang_id
)

# Decode
translated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("English:", sentence)
print("Tamil  :", translated)


AttributeError: NllbTokenizerFast has no attribute lang_code_to_id

In [58]:
tgt_lang_id = tokenizer.lang_code_to_id["tam_Taml"]
output_ids = model.generate(forced_bos_token_id=tgt_lang_id)


AttributeError: NllbTokenizerFast has no attribute lang_code_to_id

In [59]:
import transformers
print(transformers.__version__)


4.52.4


In [60]:
# Install if needed:
# pip install transformers torch

from transformers import pipeline

# Create a translation pipeline for English → Tamil
translator = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M",
    src_lang="eng_Latn",
    tgt_lang="tam_Taml"
)

# English sentence
sentence = "The accused stole money."

# Translate
translated = translator(sentence)[0]["translation_text"]

print("English:", sentence)
print("Tamil  :", translated)


Device set to use cpu


English: The accused stole money.
Tamil  : குற்றம் சாட்டப்பட்டவர் பணத்தை திருடினார்.


In [ ]:
import json
from transformers import pipeline
from sacrebleu import corpus_bleu

# -------------------------------
# Load JSON file
# -------------------------------
file_path =   # replace with your file path
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

pairs = data["pairs"]

# -------------------------------
# Translator pipeline
# -------------------------------
translator = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M",
    src_lang="eng_Latn",
    tgt_lang="tam_Taml"
)

# -------------------------------
# Prepare texts for translation and BLEU
# -------------------------------
english_texts = [pair["english"] for pair in pairs]
reference_tamil_texts = [[pair["tamil"]] for pair in pairs]  # sacrebleu expects list of list

# -------------------------------
# Translate
# -------------------------------
predicted_tamil = []
for text in english_texts:
    translated = translator(text)[0]["translation_text"]
    predicted_tamil.append(translated)

# -------------------------------
# Compute BLEU score
# -------------------------------
bleu = corpus_bleu(predicted_tamil, reference_tamil_texts)
print(f"\nBLEU score: {bleu.score:.2f}\n")

# -------------------------------
# Print predictions
# -------------------------------
for i, pair in enumerate(pairs):
    print(f"EN:   {pair['english']}")
    print(f"Ref:  {pair['tamil']}")
    print(f"Pred: {predicted_tamil[i]}")
    print("-" * 50)


Device set to use cpu


KeyError: 'tamil'

In [62]:
import json
from transformers import pipeline
from sacrebleu import corpus_bleu

# -------------------------------
# Load JSON file
# -------------------------------
file_path = "D://reserach-testing//translation_test_pairs_50.json"  # replace with your actual JSON file path

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

pairs = data.get("pairs", [])

# -------------------------------
# Filter valid entries (skip if 'tamil' or 'english' missing)
# -------------------------------
valid_pairs = [p for p in pairs if "english" in p and "tamil" in p]

if not valid_pairs:
    raise ValueError("No valid sentence pairs found in the JSON file.")

# -------------------------------
# Translator pipeline
# -------------------------------
translator = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M",
    src_lang="eng_Latn",
    tgt_lang="tam_Taml",
    device=-1  # CPU (-1), or 0 for GPU
)

# -------------------------------
# Prepare texts
# -------------------------------
english_texts = [p["english"] for p in valid_pairs]
reference_tamil_texts = [[p["tamil"]] for p in valid_pairs]  # sacrebleu expects list of list

# -------------------------------
# Translate
# -------------------------------
predicted_tamil = []
for text in english_texts:
    translated = translator(text)[0]["translation_text"]
    predicted_tamil.append(translated)

# -------------------------------
# Compute BLEU score
# -------------------------------
bleu = corpus_bleu(predicted_tamil, reference_tamil_texts)
print(f"\nBLEU score: {bleu.score:.2f}\n")

# -------------------------------
# Print predictions
# -------------------------------
for i, p in enumerate(valid_pairs):
    print(f"EN:   {p['english']}")
    print(f"Ref:  {p['tamil']}")
    print(f"Pred: {predicted_tamil[i]}")
    print("-" * 50)


Device set to use cpu



BLEU score: 10.25

EN:   Under which section of the law is cheating to obtain money considered an offence?
Ref:  ஒரு நபரை ஏமாற்றி பணம் பறிப்பது எந்த சட்டத்தின் கீழ் குற்றமாக கருதப்படுகிறது?
Pred: சட்டத்தின் எந்த பிரிவின் கீழ் பணம் சம்பாதிப்பதற்காக மோசடி செய்வது குற்றமாகக் கருதப்படுகிறது?
--------------------------------------------------
EN:   What is the punishment for the offence of cheating?
Ref:  கபடக் குற்றத்திற்கான தண்டனை என்ன?
Pred: மோசடி செய்தவருக்கு என்ன தண்டனை?
--------------------------------------------------
EN:   What offence is committed if a person obtains property by providing false information?
Ref:  ஒரு நபரிடம் தவறான தகவலை வழங்கி சொத்தைப் பெற்றால் அது எந்த குற்றம்?
Pred: பொய்யான தகவல்களை வழங்குவதன் மூலம் சொத்து வாங்கினால் என்ன குற்றம் செய்யப்படுகிறது?
--------------------------------------------------
EN:   What rights does a suspect have when he is arrested?
Ref:  குற்றவாளி கைது செய்யப்பட்டால் அவருக்கு என்ன உரிமைகள் உள்ளன?
Pred: கைது செய்யப்பட்டால் சந்தேக நபருக்கு 

In [63]:

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

pairs = data.get("pairs", [])

print(f"Total pairs in JSON: {len(pairs)}")


Total pairs in JSON: 90


In [64]:
valid_pairs = [p for p in pairs if "english" in p and "tamil" in p]
print(f"Pairs processed for translation: {len(valid_pairs)}")


Pairs processed for translation: 88


In [65]:
invalid_pairs = [p.get("id") for p in pairs if "english" not in p or "tamil" not in p]
print("Pairs skipped due to missing keys:", invalid_pairs)


Pairs skipped due to missing keys: [57, 57]


In [69]:
import json
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
import sacrebleu
import torch

# -------------------------------
# Config
# -------------------------------
json_file_path = "D://reserach-testing//translation_test_pairs_50.json" # path to your JSON with "english" & "tamil"
model_name = "facebook/m2m100_418M"
device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------------
# Load model
# -------------------------------
tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name).to(device)

# -------------------------------
# Load JSON pairs
# -------------------------------
with open(json_file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

pairs = data["pairs"]

english_texts = []
reference_tamil_texts = []
skipped_ids = []

# -------------------------------
# Prepare data
# -------------------------------
for pair in pairs:
    try:
        eng = pair["english"]
        tam = pair["tamil"]
        english_texts.append(eng)
        reference_tamil_texts.append([tam])  # sacrebleu expects list of lists
    except KeyError:
        skipped_ids.append(pair.get("id", "Unknown"))

print(f"Total pairs in JSON: {len(pairs)}")
print(f"Pairs skipped due to missing keys: {skipped_ids}")

# -------------------------------
# Translate
# -------------------------------
predicted_tamil = []
for sent in english_texts:
    tokenizer.src_lang = "en"
    inputs = tokenizer(sent, return_tensors="pt").to(device)
    generated_tokens = model.generate(**inputs, forced_bos_token_id=tokenizer.get_lang_id("ta"))
    translated = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
    predicted_tamil.append(translated)

# -------------------------------
# BLEU score
# -------------------------------
bleu = sacrebleu.corpus_bleu(predicted_tamil, reference_tamil_texts)
print("\nBLEU score:", round(bleu.score, 2))

# -------------------------------
# Print a few examples
# -------------------------------
for i in range(min(5, len(english_texts))):
    print("-" * 50)
    print("EN:", english_texts[i])
    print("Ref:", reference_tamil_texts[i][0])
    print("Pred:", predicted_tamil[i])


Total pairs in JSON: 90
Pairs skipped due to missing keys: [57, 57]

BLEU score: 9.65
--------------------------------------------------
EN: Under which section of the law is cheating to obtain money considered an offence?
Ref: ஒரு நபரை ஏமாற்றி பணம் பறிப்பது எந்த சட்டத்தின் கீழ் குற்றமாக கருதப்படுகிறது?
Pred: பணம் சம்பாதிப்பதற்கான குற்றச்சாட்டுகள் எவ்வாறு வழங்கப்படுகின்றன?
--------------------------------------------------
EN: What is the punishment for the offence of cheating?
Ref: கபடக் குற்றத்திற்கான தண்டனை என்ன?
Pred: குற்றவாளிகளுக்கு வேதனை என்ன?
--------------------------------------------------
EN: What offence is committed if a person obtains property by providing false information?
Ref: ஒரு நபரிடம் தவறான தகவலை வழங்கி சொத்தைப் பெற்றால் அது எந்த குற்றம்?
Pred: அதைவிடவரைமுறை விளக்கமும், Technical information உம் பயனுள்ளவை.
--------------------------------------------------
EN: What rights does a suspect have when he is arrested?
Ref: குற்றவாளி கைது செய்யப்பட்டால் அவருக்கு என்ன உரி

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import sacrebleu

# -------------------------------
# CONFIG
# -------------------------------
json_file_path = "D://reserach-testing//translation_test_pairs_50.json"  # Your JSON file
device = "cuda" if torch.cuda.is_available() else "cpu"

# Choose your model
# Options:
# "facebook/nllb-200-1.3B"
# "facebook/nllb-200-3.3B"
# "ai4bharat/IndicTrans2"
# "Helsinki-NLP/opus-mt-en-tam"
model_name = "facebook/nllb-200-1.3B"

# -------------------------------
# LOAD MODEL
# -------------------------------
print(f"Loading model: {model_name} ...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# For NLLB models, forced BOS ID is needed
use_forced_bos = "nllb-200" in model_name.lower()
if use_forced_bos:
    # Latest transformers: use get_lang_id
    try:
        tgt_lang_id = tokenizer.get_lang_id("tam_Taml")
    except AttributeError:
        tgt_lang_id = None
        print("Warning: could not get lang_id for Tamil. Output may be wrong.")

# -------------------------------
# LOAD JSON
# -------------------------------
with open(json_file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

pairs = data["pairs"]

english_texts = []
reference_tamil_texts = []
skipped_ids = []

for pair in pairs:
    try:
        english_texts.append(pair["english"])
        reference_tamil_texts.append([pair["tamil"]])  # sacrebleu expects list of list
    except KeyError:
        skipped_ids.append(pair.get("id", "Unknown"))

print(f"Total pairs in JSON: {len(pairs)}")
print(f"Pairs skipped due to missing keys: {skipped_ids}")

# -------------------------------
# TRANSLATE
# -------------------------------
predicted_tamil = []
for sent in english_texts:
    inputs = tokenizer(sent, return_tensors="pt").to(device)
    
    generate_kwargs = {"max_length": 100, "num_beams": 5, "early_stopping": True}
    if use_forced_bos and tgt_lang_id is not None:
        generate_kwargs["forced_bos_token_id"] = tgt_lang_id

    output_ids = model.generate(**inputs, **generate_kwargs)
    translated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    predicted_tamil.append(translated)

# -------------------------------
# BLEU SCORE
# -------------------------------
bleu = sacrebleu.corpus_bleu(predicted_tamil, reference_tamil_texts)
print("\nBLEU score:", round(bleu.score, 2))

# -------------------------------
# PRINT EXAMPLES
# -------------------------------
for i in range(min(5, len(english_texts))):
    print("-" * 50)
    print("EN:", english_texts[i])
    print("Ref:", reference_tamil_texts[i][0])
    print("Pred:", predicted_tamil[i])


Loading model: facebook/nllb-200-1.3B ...
Total pairs in JSON: 90
Pairs skipped due to missing keys: [57, 57]

BLEU score: 2.45
--------------------------------------------------
EN: Under which section of the law is cheating to obtain money considered an offence?
Ref: ஒரு நபரை ஏமாற்றி பணம் பறிப்பது எந்த சட்டத்தின் கீழ் குற்றமாக கருதப்படுகிறது?
Pred: ¿Bajo qué sección de la ley es el fraude para obtener dinero considerado un delito?
--------------------------------------------------
EN: What is the punishment for the offence of cheating?
Ref: கபடக் குற்றத்திற்கான தண்டனை என்ன?
Pred: ¿Cuál es el castigo por el delito de hacer trampa?
--------------------------------------------------
EN: What offence is committed if a person obtains property by providing false information?
Ref: ஒரு நபரிடம் தவறான தகவலை வழங்கி சொத்தைப் பெற்றால் அது எந்த குற்றம்?
Pred: Quale reato si commette se una persona ottiene una proprietà fornendo informazioni false?
--------------------------------------------------

In [3]:
from transformers import pipeline

translator = pipeline(
    "translation",
    model="facebook/nllb-200-1.3B",
    tokenizer="facebook/nllb-200-1.3B",
    device=-1,  # CPU, or 0 for GPU
    src_lang="eng_Latn",  # source language
    tgt_lang="tam_Taml"   # target language
)

sentence = "The accused stole money."
result = translator(sentence)
print("English:", sentence)
print("Tamil  :", result[0]["translation_text"])



Device set to use cpu


English: The accused stole money.
Tamil  : குற்றம் சாட்டப்பட்டவர் பணத்தை திருடினார்.


In [7]:
def translate(model_name, text, src_lang=None, tgt_lang=None):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    if src_lang:
        text = f"<<{src_lang}>> {text}"

    inputs = tokenizer(text, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang) if tgt_lang else None,
        max_length=200
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [8]:
nllb13 = "facebook/nllb-200-1.3B"

In [9]:
tam_text = "ஒரு நபரை ஏமாற்றி பணம் பறிப்பது எந்த சட்ட பிரிவின் கீழ் வருகிறது?"
eng_text = "What is the punishment for cheating under Sri Lankan Penal Code?"

In [10]:
print("🟢 NLLB 1.3B EN→TA:")
print(translate(nllb13, eng_text, tgt_lang="tam_Taml"))

🟢 NLLB 1.3B EN→TA:
இலங்கை தண்டனைச் சட்டத்தின் கீழ் மோசடி செய்தால் என்ன தண்டனை கிடைக்கும்?


In [14]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import sacrebleu

# -------------------------------
# CONFIG
# -------------------------------
json_file_path = "D://reserach-testing//translation_test_pairs_50.json"
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "facebook/nllb-200-1.3B"

# -------------------------------
# TRANSLATION FUNCTION
# -------------------------------
def translate(model_name, text, src_lang=None, tgt_lang=None):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

    if src_lang:
        text = f"<<{src_lang}>> {text}"

    inputs = tokenizer(text, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang) if tgt_lang else None,
        max_length=200,
        num_beams=5,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# -------------------------------
# LOAD JSON
# -------------------------------
with open(json_file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

pairs = data.get("pairs", [])
english_texts = []
reference_tamil_texts = []
skipped_ids = []

for idx, pair in enumerate(pairs):
    if "english" in pair and "tamil" in pair:
        english_texts.append(pair["english"])
        reference_tamil_texts.append([pair["tamil"]])
    else:
        skipped_ids.append(pair.get("id", idx))

print(f"Total pairs: {len(pairs)}")
print(f"Skipped pairs: {skipped_ids}")

# -------------------------------
# TRANSLATE ALL
# -------------------------------
predicted_tamil = []
for sent in english_texts:
    translated = translate(model_name, sent, tgt_lang="tam_Taml")
    predicted_tamil.append(translated)

# -------------------------------
# BLEU SCORE
# -------------------------------
bleu = sacrebleu.corpus_bleu(predicted_tamil, reference_tamil_texts)
print("\nCorpus BLEU score:", round(bleu.score, 2))

# -------------------------------
# PRINT EXAMPLES
# -------------------------------
print("\n--- Sample Translations ---")
for i in range(min(5, len(english_texts))):
    print("-" * 50)
    print("EN:", english_texts[i])
    print("Ref:", reference_tamil_texts[i][0])
    print("Pred:", predicted_tamil[i])


Total pairs: 90
Skipped pairs: [57, 57]

Corpus BLEU score: 18.36

--- Sample Translations ---
--------------------------------------------------
EN: Under which section of the law is cheating to obtain money considered an offence?
Ref: ஒரு நபரை ஏமாற்றி பணம் பறிப்பது எந்த சட்டத்தின் கீழ் குற்றமாக கருதப்படுகிறது?
Pred: சட்டத்தின் எந்த பிரிவின் கீழ் பணம் பெறுவதற்காக மோசடி செய்வது குற்றமாக கருதப்படுகிறது?
--------------------------------------------------
EN: What is the punishment for the offence of cheating?
Ref: கபடக் குற்றத்திற்கான தண்டனை என்ன?
Pred: மோசடி செய்த குற்றத்திற்கான தண்டனை என்ன?
--------------------------------------------------
EN: What offence is committed if a person obtains property by providing false information?
Ref: ஒரு நபரிடம் தவறான தகவலை வழங்கி சொத்தைப் பெற்றால் அது எந்த குற்றம்?
Pred: ஒரு நபர் தவறான தகவல்களை வழங்குவதன் மூலம் சொத்துக்களைப் பெற்றால் என்ன குற்றம் செய்யப்படுகிறது?
--------------------------------------------------
EN: What rights does a suspect have w

In [2]:
pip install torch torchvision torchaudio transformers accelerate peft bitsandbytes datasets sentencepiece protobuf

  Using cached peft-0.18.0-py3-none-any.whl.metadata (14 kB)
  Using cached bitsandbytes-0.48.2-py3-none-win_amd64.whl.metadata (10 kB)
  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
Using cached peft-0.18.0-py3-none-any.whl (556 kB)
Using cached bitsandbytes-0.48.2-py3-none-win_amd64.whl (59.0 MB)
Using cached fsspec-2025.3.0-py3-none-any.whl (193 kB)

  Attempting uninstall: fsspec

    Found existing installation: fsspec 2025.5.1

    Uninstalling fsspec-2025.5.1:

   ---------------------------------------- 0/3 [fsspec]
      Successfully uninstalled fsspec-2025.5.1
   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ---

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
s3fs 2024.6.1 requires fsspec==2024.6.1.*, but you have fsspec 2025.3.0 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import torch
import json
import os
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM, 
    AutoTokenizer, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType

# ==========================================
# 1. SETUP
# ==========================================
model_id = "facebook/nllb-200-distilled-600M"
output_dir = "./nllb-legal-checkpoints"
final_model_dir = "./nllb-legal-final"
dataset_file = "nllb_bidir_dataset1.jsonl"

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"=== Hardware: {device.upper()} ===")

# ==========================================
# 2. DATA LOADING
# ==========================================
def load_jsonl(filename):
    data = []
    if not os.path.exists(filename):
        print(f"❌ Error: {filename} not found.")
        return []
    
    print(f"Reading {filename}...")
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if line.endswith(','): line = line[:-1]
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return data

raw_data = load_jsonl(dataset_file)
if not raw_data: exit()

full_dataset = Dataset.from_list(raw_data)
dataset = full_dataset.train_test_split(test_size=0.1, seed=42) if len(full_dataset) > 10 else {"train": full_dataset, "test": full_dataset}

# ==========================================
# 3. MODEL SETUP (Standard LoRA)
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model in float16 (Standard LoRA)...")
# We load in torch.float16 to save memory, but not 4-bit
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)

# Enable gradient checkpointing to save VRAM
model.gradient_checkpointing_enable()

# LoRA Configuration
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, 
    inference_mode=False, 
    r=32, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"] 
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# ==========================================
# 4. PREPROCESSING
# ==========================================
def preprocess_function(examples):
    inputs = [ex if ex else "" for ex in examples["src"]]
    targets = [ex if ex else "" for ex in examples["tgt"]]
    
    # Set source language for tokenizer (NLLB requires this)
    # Note: If your dataset has different languages per row, we loop.
    # But for speed in map, we usually assume a batch shares logic or handle it carefully.
    # Here is a robust way for NLLB:
    
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset["train"].map(preprocess_function, batched=True)
eval_dataset = dataset["test"].map(preprocess_function, batched=True) if "test" in dataset else None

# ==========================================
# 5. TRAINING
# ==========================================
args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    save_strategy="steps", 
    save_steps=100,               
    save_total_limit=2,          
    per_device_train_batch_size=4, # Keep small for 4GB VRAM
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,          
    fp16=True if device == "cuda" else False,
    logging_steps=10,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    tokenizer=tokenizer,
)

print("🚀 Starting Standard LoRA Training...")
trainer.train()

print("💾 Saving model...")
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print("✅ Done!")

=== Hardware: CUDA ===
Reading nllb_bidir_dataset1.jsonl...
Loading model in float16 (Standard LoRA)...


ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

In [6]:
pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
pip install transformers==4.46.3 accelerate

  Using cached transformers-4.46.3-py3-none-any.whl.metadata (44 kB)
  Using cached tokenizers-0.20.3-cp312-none-win_amd64.whl.metadata (6.9 kB)
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.0 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.0 MB 1.3 MB/s eta 0:00:08
   -- ------------------------------------- 0.5/10.0 MB 1.3 MB/s eta 0:00:08
   --- ------------------------------------ 0.8/10.0 MB 1.2 MB/s eta 0:00:08
   ----- ---------------------------------- 1.3/10.0 MB 1.3 MB/s eta 0:00:07
   ------ --------------------------------- 1.6/10.0 MB 1.4 MB/s eta 0:00:07
   ------- -------------------------------- 1.8/10.0 MB 1.3 MB/s eta 0:00:07
   -------- ------------------------------- 2.1/10.0 MB 1.4 MB/s eta 0:00:06
   ---------- ----------------------------- 2.6/10.0 MB 1.4 MB/s eta 0:00:06
   ------------ --------------------------- 3.1/10.0 MB 1.6 MB/s eta 0:00:05
   ------------- -----

  You can safely remove it manually.

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
pip uninstall -y transformers

Found existing installation: transformers 4.46.3
Uninstalling transformers-4.46.3:
  Successfully uninstalled transformers-4.46.3
Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install transformers==4.46.3 accelerate

  Using cached transformers-4.46.3-py3-none-any.whl.metadata (44 kB)
Using cached transformers-4.46.3-py3-none-any.whl (10.0 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 4.52.4
    Uninstalling transformers-4.52.4:
      Successfully uninstalled transformers-4.52.4
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys
import torch
import transformers
import peft
import accelerate

print("=== 🔍 Environment Check ===")
print(f"Python Path:     {sys.executable}")
print(f"Torch Version:   {torch.__version__}")
print(f"Transformers:    {transformers.__version__}")
print(f"PEFT Version:    {peft.__version__}")
print(f"Accelerate:      {accelerate.__version__}")

print("\n=== 🎯 Verification ===")
if "venv" in sys.executable:
    print("✅ Correct: You are using the Virtual Environment (venv).")
else:
    print("⚠️  WARNING: You seem to be using the global Anaconda environment, NOT your venv!")
    print("    If versions don't match, you need to select the correct Kernel in the notebook.")

d:\reserach-testing\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== 🔍 Environment Check ===
Python Path:     d:\reserach-testing\venv\Scripts\python.exe
Torch Version:   2.5.1+cu121
Transformers:    4.46.3
PEFT Version:    0.13.2
Accelerate:      1.0.1

=== 🎯 Verification ===
✅ Correct: You are using the Virtual Environment (venv).


In [1]:
import torch
import json
import os
import sys
import warnings
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM, 
    AutoTokenizer, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType

# Suppress warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
model_id = "facebook/nllb-200-distilled-600M"
output_dir = "./nllb-legal-checkpoints"
final_model_dir = "./nllb-legal-final"
dataset_file = "nllb_bidir_dataset1.jsonl"

print("=== Hardware Check ===")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Training on: {device.upper()}")

# ==========================================
# 2. DATA LOADING (Safe & Clean)
# ==========================================
def load_and_clean_jsonl(filename):
    data = []
    if not os.path.exists(filename):
        print(f"❌ Error: Dataset file '{filename}' not found.")
        sys.exit()
        
    print(f"Reading {filename}...")
    with open(filename, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line: continue
            if line.endswith(','): line = line[:-1]
                
            try:
                entry = json.loads(line)
                # Validation: Ensure all columns exist and are not empty
                if not all(k in entry for k in ("src", "tgt", "src_lang", "tgt_lang")):
                    continue
                if not entry["src"] or not entry["tgt"]:
                    continue
                data.append(entry)
            except json.JSONDecodeError:
                continue
                
    print(f"✅ Loaded {len(data)} valid rows.")
    return data

raw_data = load_and_clean_jsonl(dataset_file)
if len(raw_data) == 0:
    print("❌ Critical Error: Dataset is empty.")
    sys.exit()

full_dataset = Dataset.from_list(raw_data)
if len(full_dataset) > 10:
    dataset = full_dataset.train_test_split(test_size=0.1, seed=42)
else:
    dataset = {"train": full_dataset, "test": full_dataset}

# ==========================================
# 3. MODEL & TOKENIZER (The Fix)
# ==========================================
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# FIX: Removed device_map="auto" to prevent meta tensor errors
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    low_cpu_mem_usage=True
)

# Manually move to GPU
if device == "cuda":
    model = model.to(device)

model.gradient_checkpointing_enable()

# LoRA Configuration
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, 
    inference_mode=False, 
    r=32, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"] 
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# ==========================================
# 4. PREPROCESSING (Single Row Logic)
# ==========================================
def preprocess_single_example(example):
    """Processes ONE row at a time to handle language switching correctly."""
    src_text = example["src"]
    tgt_text = example["tgt"]
    src_lang = example["src_lang"]
    tgt_lang = example["tgt_lang"]

    try:
        # 1. Source Language
        tokenizer.src_lang = src_lang
        model_inputs = tokenizer(src_text, max_length=128, truncation=True)
        
        # 2. Target Language
        tokenizer.src_lang = tgt_lang
        labels = tokenizer(text_target=tgt_text, max_length=128, truncation=True)

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    except Exception:
        return None

print("⏳ Tokenizing dataset (One-by-one mode)...")
tokenized_datasets = dataset["train"].map(
    preprocess_single_example, 
    batched=False,
    remove_columns=dataset["train"].column_names,
    desc="Processing Train Data"
).filter(lambda x: x is not None)

if "test" in dataset:
    eval_dataset = dataset["test"].map(
        preprocess_single_example, 
        batched=False,
        remove_columns=dataset["test"].column_names,
        desc="Processing Test Data"
    ).filter(lambda x: x is not None)
else:
    eval_dataset = None

print(f"✅ Processing Complete. Training samples: {len(tokenized_datasets)}")

# ==========================================
# 5. TRAINING
# ==========================================
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    save_strategy="steps", 
    save_steps=100,               
    save_total_limit=2,          
    per_device_train_batch_size=4, 
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,          
    fp16=True if device == "cuda" else False,
    logging_steps=10,
    report_to="none",
    
    # CRITICAL FIX: Stops Trainer from deleting columns it "thinks" are unused
    remove_unused_columns=False 
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("🚀 Starting Training...")
trainer.train()

print("💾 Saving final model...")
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print("✅ DONE! Model saved successfully.")

d:\reserach-testing\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Hardware Check ===
✅ Training on: CUDA
Reading nllb_bidir_dataset1.jsonl...
✅ Loaded 8684 valid rows.
Loading model...
trainable params: 17,301,504 || all params: 632,375,296 || trainable%: 2.7360
⏳ Tokenizing dataset (One-by-one mode)...


Filter: 100%|██████████| 869/869 [00:00<00:00, 74412.03 examples/s]


✅ Processing Complete. Training samples: 7815
🚀 Starting Training...


  0%|          | 0/1464 [00:00<?, ?it/s]

ValueError: You should supply an encoding or a list of encodings to this method that includes input_ids, but you provided ['src', 'tgt', 'src_lang', 'tgt_lang']

In [2]:
import torch
import json
import os
import sys
import warnings
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM, 
    AutoTokenizer, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType

# Suppress warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
model_id = "facebook/nllb-200-distilled-600M"
output_dir = "./nllb-legal-checkpoints"
final_model_dir = "./nllb-legal-final"
dataset_file = "nllb_bidir_dataset1.jsonl"

print("=== Hardware Check ===")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Training on: {device.upper()}")

# ==========================================
# 2. DATA LOADING (Safe & Clean)
# ==========================================
def load_and_clean_jsonl(filename):
    data = []
    if not os.path.exists(filename):
        print(f"❌ Error: Dataset file '{filename}' not found.")
        sys.exit()
        
    print(f"Reading {filename}...")
    with open(filename, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line: continue
            if line.endswith(','): line = line[:-1]
                
            try:
                entry = json.loads(line)
                # Validation: Ensure all columns exist and are not empty
                if not all(k in entry for k in ("src", "tgt", "src_lang", "tgt_lang")):
                    continue
                if not entry["src"] or not entry["tgt"]:
                    continue
                data.append(entry)
            except json.JSONDecodeError:
                continue
                
    print(f"✅ Loaded {len(data)} valid rows.")
    return data

raw_data = load_and_clean_jsonl(dataset_file)
if len(raw_data) == 0:
    print("❌ Critical Error: Dataset is empty.")
    sys.exit()

full_dataset = Dataset.from_list(raw_data)
if len(full_dataset) > 10:
    dataset = full_dataset.train_test_split(test_size=0.1, seed=42)
else:
    dataset = {"train": full_dataset, "test": full_dataset}

# ==========================================
# 3. MODEL & TOKENIZER (Meta Tensor Fix)
# ==========================================
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# FIX: Load manually to avoid Windows "Meta Tensor" errors
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    low_cpu_mem_usage=True
)

# Manually move to GPU
if device == "cuda":
    model = model.to(device)

model.gradient_checkpointing_enable()

# LoRA Configuration
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, 
    inference_mode=False, 
    r=32, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"] 
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# ==========================================
# 4. PREPROCESSING (Robust & Cache-Free)
# ==========================================
def preprocess_single_example(example):
    """Processes ONE row at a time to handle language switching correctly."""
    src_text = example["src"]
    tgt_text = example["tgt"]
    src_lang = example["src_lang"]
    tgt_lang = example["tgt_lang"]

    try:
        # 1. Source Language
        tokenizer.src_lang = src_lang
        model_inputs = tokenizer(src_text, max_length=128, truncation=True)
        
        # 2. Target Language
        tokenizer.src_lang = tgt_lang
        labels = tokenizer(text_target=tgt_text, max_length=128, truncation=True)

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    except Exception:
        return None

print("⏳ Tokenizing dataset (Fresh Run - No Cache)...")

# 1. Map and Process
tokenized_datasets = dataset["train"].map(
    preprocess_single_example, 
    batched=False,
    load_from_cache_file=False, # CRITICAL: Force fresh processing
    desc="Processing Train Data"
).filter(lambda x: x is not None)

if "test" in dataset:
    eval_dataset = dataset["test"].map(
        preprocess_single_example, 
        batched=False,
        load_from_cache_file=False, # CRITICAL: Force fresh processing
        desc="Processing Test Data"
    ).filter(lambda x: x is not None)
else:
    eval_dataset = None

# 2. MANUAL CLEANUP (Force remove old columns)
# This prevents the "You provided ['src', 'tgt']" error
cols_to_remove = ["src", "tgt", "src_lang", "tgt_lang"]
existing_cols = tokenized_datasets.column_names
cols_to_drop = [c for c in cols_to_remove if c in existing_cols]

if cols_to_drop:
    print(f"🧹 Manually removing old columns: {cols_to_drop}")
    tokenized_datasets = tokenized_datasets.remove_columns(cols_to_drop)
    if eval_dataset:
        eval_dataset = eval_dataset.remove_columns(cols_to_drop)

# 3. FORCE PYTORCH FORMAT
# This guarantees the Trainer sees Tensors, not lists
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
if eval_dataset:
    eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(f"✅ Final Columns: {tokenized_datasets.column_names}")
print(f"✅ Processing Complete. Training samples: {len(tokenized_datasets)}")

# ==========================================
# 5. TRAINING
# ==========================================
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    save_strategy="steps", 
    save_steps=100,               
    save_total_limit=2,          
    per_device_train_batch_size=4, 
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,          
    fp16=True if device == "cuda" else False,
    logging_steps=10,
    report_to="none",
    
    # CRITICAL FIX: Stops Trainer from deleting columns it "thinks" are unused
    remove_unused_columns=False 
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("🚀 Starting Training...")
trainer.train()

print("💾 Saving final model...")
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print("✅ DONE! Model saved successfully.")

=== Hardware Check ===
✅ Training on: CUDA
Reading nllb_bidir_dataset1.jsonl...
✅ Loaded 8684 valid rows.
Loading model...
trainable params: 17,301,504 || all params: 632,375,296 || trainable%: 2.7360
⏳ Tokenizing dataset (Fresh Run - No Cache)...


Filter: 100%|██████████| 869/869 [00:00<00:00, 50850.33 examples/s]


🧹 Manually removing old columns: ['src', 'tgt', 'src_lang', 'tgt_lang']


ValueError: Columns ['attention_mask', 'input_ids', 'labels'] not in the dataset. Current columns in the dataset: []

In [3]:
import torch
import json
import os
import sys
import warnings
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM, 
    AutoTokenizer, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType

warnings.filterwarnings("ignore")

# ==========================================
# 1. CONFIGURATION
# ==========================================
model_id = "facebook/nllb-200-distilled-600M"
output_dir = "./nllb-legal-checkpoints"
final_model_dir = "./nllb-legal-final"
dataset_file = "nllb_bidir_dataset1.jsonl"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Hardware: {device.upper()}")

# ==========================================
# 2. DATA LOADING
# ==========================================
def load_jsonl(filename):
    data = []
    if not os.path.exists(filename):
        print(f"❌ Error: {filename} not found.")
        sys.exit()
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                line = line.strip()
                if not line: continue
                if line.endswith(','): line = line[:-1]
                entry = json.loads(line)
                if "src" in entry and "tgt" in entry and "src_lang" in entry and "tgt_lang" in entry:
                    data.append(entry)
            except:
                continue
    return data

raw_data = load_jsonl(dataset_file)
if not raw_data:
    print("❌ Error: Dataset is empty.")
    sys.exit()

print(f"✅ Loaded {len(raw_data)} rows.")
full_dataset = Dataset.from_list(raw_data)
dataset = full_dataset.train_test_split(test_size=0.1, seed=42) if len(full_dataset) > 10 else {"train": full_dataset, "test": full_dataset}

# ==========================================
# 3. TOKENIZER TEST (Crucial Debug Step)
# ==========================================
print("\n🔍 --- TESTING TOKENIZER ON ONE ROW ---")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Grab the first row from the dataset
test_row = dataset["train"][0]
print(f"Test Input: {test_row}")

try:
    # 1. Test Source Language Setting
    print(f"Attempting to set src_lang to: '{test_row['src_lang']}'")
    tokenizer.src_lang = test_row["src_lang"]
    
    # 2. Test Tokenization
    test_input = tokenizer(test_row["src"], max_length=128, truncation=True)
    print("✅ Input Tokenization: SUCCESS")
    
    # 3. Test Target Language
    print(f"Attempting to set tgt_lang to: '{test_row['tgt_lang']}'")
    tokenizer.src_lang = test_row["tgt_lang"]
    test_target = tokenizer(text_target=test_row["tgt"], max_length=128, truncation=True)
    print("✅ Target Tokenization: SUCCESS")
    
except Exception as e:
    print(f"\n❌ CRITICAL ERROR DURING TEST: {e}")
    print("💡 HINT: If the error mentions 'vocabulary', your language code (e.g., 'eng_Latn') might be wrong.")
    print("💡 STOPPING HERE TO FIX THE DATA.")
    sys.exit()

# ==========================================
# 4. FULL PROCESSING (If test passes)
# ==========================================
print("\n⏳ Processing full dataset...")

def preprocess_function(example):
    try:
        tokenizer.src_lang = example["src_lang"]
        inputs = tokenizer(example["src"], max_length=128, truncation=True)
        
        tokenizer.src_lang = example["tgt_lang"]
        targets = tokenizer(text_target=example["tgt"], max_length=128, truncation=True)
        
        inputs["labels"] = targets["input_ids"]
        return inputs
    except:
        return {"input_ids": []} # Return empty if fail

# Map without removing columns yet (safer)
tokenized_datasets = dataset["train"].map(
    preprocess_function, 
    batched=False,
    load_from_cache_file=False
)

if "test" in dataset:
    eval_dataset = dataset["test"].map(preprocess_function, batched=False, load_from_cache_file=False)
else:
    eval_dataset = None

# Filter out failed rows (empty input_ids)
print(f"Rows before filtering: {len(tokenized_datasets)}")
tokenized_datasets = tokenized_datasets.filter(lambda x: len(x["input_ids"]) > 0)
print(f"Rows after filtering: {len(tokenized_datasets)}")

if len(tokenized_datasets) == 0:
    print("❌ Error: All rows failed processing. Check the logs above.")
    sys.exit()

# Set format
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
if eval_dataset:
    eval_dataset = eval_dataset.filter(lambda x: len(x["input_ids"]) > 0)
    eval_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# ==========================================
# 5. MODEL & TRAINING
# ==========================================
print("Loading model...")
model = AutoModelForSeq2SeqLM.from_pretrained(model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True)
model = model.to(device)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, inference_mode=False, r=32, lora_alpha=32, lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    save_strategy="steps", save_steps=100, save_total_limit=2,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    learning_rate=2e-4, num_train_epochs=3, fp16=(device=="cuda"),
    logging_steps=10, report_to="none", remove_unused_columns=False
)

trainer = Seq2SeqTrainer(
    model=model, args=args, train_dataset=tokenized_datasets,
    eval_dataset=eval_dataset, data_collator=data_collator, tokenizer=tokenizer
)

print("🚀 Starting Training...")
trainer.train()
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print("✅ Done!")

✅ Hardware: CUDA
✅ Loaded 8684 rows.

🔍 --- TESTING TOKENIZER ON ONE ROW ---
Test Input: {'src': 'பிணை மனுவுக்கு எதிர்ப்பு தெரிவிக்கும் நடைமுறை என்ன?', 'tgt': 'What is the procedure for filing objections to a bail application?', 'src_lang': 'tam_Taml', 'tgt_lang': 'eng_Latn'}
Attempting to set src_lang to: 'tam_Taml'
✅ Input Tokenization: SUCCESS
Attempting to set tgt_lang to: 'eng_Latn'

❌ CRITICAL ERROR DURING TEST: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'
💡 HINT: If the error mentions 'vocabulary', your language code (e.g., 'eng_Latn') might be wrong.
💡 STOPPING HERE TO FIX THE DATA.


SystemExit: 

In [4]:
import torch
import json
import os
import sys
import warnings
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM, 
    AutoTokenizer, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType

warnings.filterwarnings("ignore")

# ==========================================
# 1. SETUP
# ==========================================
model_id = "facebook/nllb-200-distilled-600M"
output_dir = "./nllb-legal-checkpoints"
final_model_dir = "./nllb-legal-final"
dataset_file = "nllb_bidir_dataset1.jsonl"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Hardware: {device.upper()}")

# ==========================================
# 2. DATA LOADING (With Aggressive Cleaning)
# ==========================================
def load_clean_data(filename):
    data = []
    if not os.path.exists(filename):
        print(f"❌ Error: {filename} not found.")
        sys.exit()
        
    print(f"Reading {filename}...")
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if line.endswith(','): line = line[:-1]
                
            try:
                entry = json.loads(line)
                
                # Check for required keys
                if not all(k in entry for k in ("src", "tgt", "src_lang", "tgt_lang")):
                    continue
                    
                # AGGRESSIVE CLEANING (The Fix)
                # .strip() removes hidden newlines/spaces that cause the crash
                src_lang = entry["src_lang"].strip()
                tgt_lang = entry["tgt_lang"].strip()
                
                # Validate lang codes are not empty
                if not src_lang or not tgt_lang:
                    continue
                    
                entry["src_lang"] = src_lang
                entry["tgt_lang"] = tgt_lang
                data.append(entry)
            except:
                continue
                
    print(f"✅ Loaded {len(data)} cleaned rows.")
    return data

raw_data = load_clean_data(dataset_file)
if not raw_data:
    print("❌ Dataset empty.")
    sys.exit()

full_dataset = Dataset.from_list(raw_data)
dataset = full_dataset.train_test_split(test_size=0.1, seed=42) if len(full_dataset) > 10 else {"train": full_dataset, "test": full_dataset}

# ==========================================
# 3. DEBUG: VERIFY LANGUAGE CODES
# ==========================================
print("\n🔍 --- DEBUG: CHECKING LANGUAGE CODES ---")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Test the first row's language codes explicitly
sample_row = dataset["train"][0]
s_lang = sample_row["src_lang"]
t_lang = sample_row["tgt_lang"]

print(f"Sample Row Langs: Source='{s_lang}', Target='{t_lang}'")

# Check if Tokenizer recognizes them
s_id = tokenizer.convert_tokens_to_ids(s_lang)
t_id = tokenizer.convert_tokens_to_ids(t_lang)

print(f"ID for '{s_lang}': {s_id}")
print(f"ID for '{t_lang}': {t_id}")

# CRITICAL CHECK
if s_id == tokenizer.unk_token_id or t_id == tokenizer.unk_token_id:
    print("❌ FATAL ERROR: The tokenizer does not recognize your language code!")
    print("   It returned 'unk_token_id'. Check for typos in your JSON.")
    print("   Example valid codes: 'eng_Latn', 'tam_Taml'.")
    sys.exit()
elif s_id is None or t_id is None:
    print("❌ FATAL ERROR: Tokenizer returned None for a language ID.")
    sys.exit()
else:
    print("✅ Language codes are VALID and recognized by the tokenizer.")

# ==========================================
# 4. PROCESSING
# ==========================================
def preprocess_function(example):
    try:
        # 1. Source
        tokenizer.src_lang = example["src_lang"]
        inputs = tokenizer(example["src"], max_length=128, truncation=True)
        
        # 2. Target (Manual Tokenization to avoid API bugs)
        tokenizer.src_lang = example["tgt_lang"]
        # We tokenize target as regular input, then assign to labels
        targets = tokenizer(example["tgt"], max_length=128, truncation=True)
        
        inputs["labels"] = targets["input_ids"]
        return inputs
    except:
        return None

print("⏳ Tokenizing...")
tokenized_datasets = dataset["train"].map(
    preprocess_function, 
    batched=False, 
    remove_columns=dataset["train"].column_names,
    load_from_cache_file=False
).filter(lambda x: x is not None)

if "test" in dataset:
    eval_dataset = dataset["test"].map(
        preprocess_function, 
        batched=False, 
        remove_columns=dataset["test"].column_names,
        load_from_cache_file=False
    ).filter(lambda x: x is not None)
else:
    eval_dataset = None

# Set Format
tokenized_datasets.set_format("torch")
if eval_dataset: eval_dataset.set_format("torch")

print(f"✅ Training data shape: {len(tokenized_datasets)} rows")

# ==========================================
# 5. TRAINING
# ==========================================
print("Loading model...")
# Manual Load
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to(device)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, inference_mode=False, r=32, lora_alpha=32, lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    save_strategy="steps", save_steps=100, save_total_limit=2,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    learning_rate=2e-4, num_train_epochs=3, fp16=True,
    logging_steps=10, report_to="none", remove_unused_columns=False
)

trainer = Seq2SeqTrainer(
    model=model, args=args, train_dataset=tokenized_datasets,
    eval_dataset=eval_dataset, data_collator=data_collator, tokenizer=tokenizer
)

print("🚀 Starting Training...")
trainer.train()

print("💾 Saving...")
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print("✅ Done!")

✅ Hardware: CUDA
Reading nllb_bidir_dataset1.jsonl...
✅ Loaded 8684 cleaned rows.

🔍 --- DEBUG: CHECKING LANGUAGE CODES ---
Sample Row Langs: Source='tam_Taml', Target='eng_Latn'
ID for 'tam_Taml': 256170
ID for 'eng_Latn': 256047
✅ Language codes are VALID and recognized by the tokenizer.
⏳ Tokenizing...


Filter: 100%|██████████| 869/869 [00:00<00:00, 28144.26 examples/s]


✅ Training data shape: 7815 rows
Loading model...


  0%|          | 0/1464 [04:07<?, ?it/s]

trainable params: 17,301,504 || all params: 632,375,296 || trainable%: 2.7360
🚀 Starting Training...


...

{'loss': 0.8068, 'grad_norm': 3.4986586570739746, 'learning_rate': 1.912568306010929e-06, 'epoch': 2.97}


100%|█████████▉| 1460/1464 [3:37:35<00:15,  3.75s/it]

{'loss': 0.6962, 'grad_norm': 2.9324004650115967, 'learning_rate': 5.46448087431694e-07, 'epoch': 2.99}


100%|██████████| 1464/1464 [3:37:53<00:00,  8.93s/it]


{'train_runtime': 13073.7478, 'train_samples_per_second': 1.793, 'train_steps_per_second': 0.112, 'train_loss': 0.7965028082412449, 'epoch': 3.0}
💾 Saving...
✅ Done!


In [5]:
import shutil
import os

# The folder where your model was saved
source_folder = "./nllb-legal-final"
# The name of the zip file you want to create
output_filename = "nllb-legal-model"

# Check if folder exists first
if os.path.exists(source_folder):
    print(f"📦 Zipping '{source_folder}'...")
    shutil.make_archive(output_filename, 'zip', source_folder)
    print(f"✅ Success! Created: {output_filename}.zip")
else:
    print(f"❌ Error: Folder '{source_folder}' not found. Did training finish?")

📦 Zipping './nllb-legal-final'...
✅ Success! Created: nllb-legal-model.zip


In [4]:
import os
import torch
import json
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel
import evaluate

# ==========================================
# 1. SETUP (CPU MODE)
# ==========================================
device = "cpu"
print(f"=== 🛡️ Checking BLEU Score on {device.upper()} (Robust Mode) ===")

base_model_id = "facebook/nllb-200-distilled-600M"
adapter_path = "./nllb-legal-final"
test_file = "translation_test_pairs_50.json"

# ==========================================
# 2. LOAD MODEL
# ==========================================
print("⏳ Loading Model...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(base_model_id)
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

bleu = evaluate.load("sacrebleu")

# ==========================================
# 3. ROBUST TRANSLATION FUNCTION
# ==========================================
def get_bleu_score(data, src_key, tgt_key, src_code, tgt_code):
    print(f"\n📊 Evaluating: {src_key} -> {tgt_key} ...")
    
    predictions = []
    references = []
    
    # Process each sentence
    for i, item in enumerate(data):
        # --- SAFETY CHECK ---
        # If the keys are missing, print the error and SKIP the row
        if src_key not in item:
            print(f"⚠️ SKIPPING Row {i}: Missing key '{src_key}'. Found: {list(item.keys())}")
            continue
        if tgt_key not in item:
            print(f"⚠️ SKIPPING Row {i}: Missing key '{tgt_key}'. Found: {list(item.keys())}")
            continue
            
        src_text = item[src_key]
        tgt_text = item[tgt_key]
        
        # 1. Tokenize
        tokenizer.src_lang = src_code
        inputs = tokenizer(src_text, return_tensors="pt", padding=False, truncation=True, max_length=128)
        
        # 2. Translate
        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_code),
                max_length=128,
                num_beams=4,
                early_stopping=True
            )
            
        # 3. Decode
        pred = tokenizer.decode(generated[0], skip_special_tokens=True)
        predictions.append(pred)
        references.append([tgt_text])
        
        if i % 10 == 0: print(f"   Processed {i}/{len(data)}")

    # 4. Calculate Score
    if len(predictions) > 0:
        results = bleu.compute(predictions=predictions, references=references)
        print(f"✅ BLEU Score: {results['score']:.2f}")
        return results['score']
    else:
        print("❌ No valid predictions generated.")
        return 0.0

# ==========================================
# 4. RUN CHECK
# ==========================================
if not os.path.exists(test_file):
    print(f"❌ Error: {test_file} not found.")
else:
    with open(test_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        # Handle cases where data is a list directly OR a dict with "pairs"
        if isinstance(data, dict) and "pairs" in data:
            pairs = data['pairs']
        elif isinstance(data, list):
            pairs = data
        else:
            print("❌ JSON structure unknown. Expected list or {'pairs': [...]}")
            pairs = []

    if len(pairs) > 0:
        print(f"✅ Loaded {len(pairs)} test sentences.")

        # 1. Tamil -> English
        score_te = get_bleu_score(pairs, "tamil", "english", "tam_Taml", "eng_Latn")

        # 2. English -> Tamil
        score_et = get_bleu_score(pairs, "english", "tamil", "eng_Latn", "tam_Taml")

        print("\n" + "="*30)
        print(f"🏆 FINAL SCORES")
        print(f"Tamil -> English: {score_te:.2f}")
        print(f"English -> Tamil: {score_et:.2f}")
        print("="*30)

=== 🛡️ Checking BLEU Score on CPU (Robust Mode) ===
⏳ Loading Model...
✅ Loaded 60 test sentences.

📊 Evaluating: tamil -> english ...
   Processed 0/60
   Processed 10/60
   Processed 20/60
   Processed 30/60
   Processed 40/60
   Processed 50/60
✅ BLEU Score: 37.90

📊 Evaluating: english -> tamil ...
   Processed 0/60
   Processed 10/60
   Processed 20/60
   Processed 30/60
   Processed 40/60
   Processed 50/60
✅ BLEU Score: 20.69

🏆 FINAL SCORES
Tamil -> English: 37.90
English -> Tamil: 20.69


In [4]:
# ==========================================
# AFTER TRAINING – SEQUENTIAL BIDIRECTIONAL EVALUATION
# ==========================================
import time
import json
import torch
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

# ==========================================
# DEVICE SETUP (CUDA → CPU FALLBACK)
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    print(f"🚀 Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("🛡️ Using CPU")

# ==========================================
# CONFIG
# ==========================================
base_model_id = "facebook/nllb-200-distilled-600M"
adapter_path = "./nllb-legal-final"
test_file = "translation_test_pairs_50.json"

# ==========================================
# LOAD MODEL (BASE + LoRA)
# ==========================================
print("⏳ Loading LoRA fine-tuned model...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_id)
model = PeftModel.from_pretrained(base_model, adapter_path)

model = model.to(device)
model.eval()

bleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

# ==========================================
# LOAD & NORMALIZE DATA
# ==========================================
with open(test_file, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

if isinstance(raw_data, dict) and "pairs" in raw_data:
    data = raw_data["pairs"]
elif isinstance(raw_data, list):
    data = raw_data
else:
    raise ValueError("❌ Unsupported JSON structure")

print(f"✅ Loaded {len(data)} test samples")

# ==========================================================
# 1️⃣ FULL DATASET EVALUATION: TAMIL → ENGLISH
# ==========================================================
print("\n==============================")
print("🔁 FULL EVALUATION: Tamil → English")
print("==============================")

SRC_KEY = "tamil"
TGT_KEY = "english"
SRC_LANG = "tam_Taml"
TGT_LANG = "eng_Latn"

predictions = []
references = []
total_time = 0.0
valid_count = 0

for row in data:
    if not isinstance(row, dict):
        continue
    if SRC_KEY not in row or TGT_KEY not in row:
        continue

    tokenizer.src_lang = SRC_LANG
    inputs = tokenizer(
        row[SRC_KEY],
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    if device == "cuda":
        torch.cuda.synchronize()
    start = time.time()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(TGT_LANG),
            max_length=128,
            num_beams=4,
            use_cache=True
        )

    if device == "cuda":
        torch.cuda.synchronize()
    end = time.time()

    total_time += (end - start)

    pred = tokenizer.decode(output[0], skip_special_tokens=True)
    predictions.append(pred)
    references.append([row[TGT_KEY]])
    valid_count += 1

bleu_te = bleu.compute(predictions=predictions, references=references)["score"]
chrf_te = chrf.compute(predictions=predictions, references=references)["score"]
latency_te = total_time / valid_count

print(f"Samples Evaluated : {valid_count}")
print(f"BLEU              : {bleu_te:.2f}")
print(f"CHRF++            : {chrf_te:.2f}")
print(f"Avg Latency (sec) : {latency_te:.3f}")

# ==========================================================
# 2️⃣ FULL DATASET EVALUATION: ENGLISH → TAMIL
# ==========================================================
print("\n==============================")
print("🔁 FULL EVALUATION: English → Tamil")
print("==============================")

SRC_KEY = "english"
TGT_KEY = "tamil"
SRC_LANG = "eng_Latn"
TGT_LANG = "tam_Taml"

predictions = []
references = []
total_time = 0.0
valid_count = 0

for row in data:
    if not isinstance(row, dict):
        continue
    if SRC_KEY not in row or TGT_KEY not in row:
        continue

    tokenizer.src_lang = SRC_LANG
    inputs = tokenizer(
        row[SRC_KEY],
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    if device == "cuda":
        torch.cuda.synchronize()
    start = time.time()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(TGT_LANG),
            max_length=128,
            num_beams=4,
            use_cache=True
        )

    if device == "cuda":
        torch.cuda.synchronize()
    end = time.time()

    total_time += (end - start)

    pred = tokenizer.decode(output[0], skip_special_tokens=True)
    predictions.append(pred)
    references.append([row[TGT_KEY]])
    valid_count += 1

bleu_et = bleu.compute(predictions=predictions, references=references)["score"]
chrf_et = chrf.compute(predictions=predictions, references=references)["score"]
latency_et = total_time / valid_count

print(f"Samples Evaluated : {valid_count}")
print(f"BLEU              : {bleu_et:.2f}")
print(f"CHRF++            : {chrf_et:.2f}")
print(f"Avg Latency (sec) : {latency_et:.3f}")

# ==========================================================
# FINAL SUMMARY
# ==========================================================
print("\n==============================")
print("🏁 FINAL BIDIRECTIONAL SUMMARY")
print("==============================")
print(f"Tamil → English | BLEU: {bleu_te:.2f} | CHRF++: {chrf_te:.2f}")
print(f"English → Tamil | BLEU: {bleu_et:.2f} | CHRF++: {chrf_et:.2f}")
print("==============================")


🚀 Using GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
⏳ Loading LoRA fine-tuned model...
✅ Loaded 60 test samples

🔁 FULL EVALUATION: Tamil → English
Samples Evaluated : 60
BLEU              : 38.74
CHRF++            : 63.47
Avg Latency (sec) : 2.088

🔁 FULL EVALUATION: English → Tamil
Samples Evaluated : 60
BLEU              : 20.58
CHRF++            : 65.10
Avg Latency (sec) : 2.576

🏁 FINAL BIDIRECTIONAL SUMMARY
Tamil → English | BLEU: 38.74 | CHRF++: 63.47
English → Tamil | BLEU: 20.58 | CHRF++: 65.10
